# Ecowitt — schema explorer and hourly metric table

Samples every table and view **discovered at runtime** from the catalog, then
builds an hour × metric table for yesterday and today.

**Prerequisite:** the SSH tunnel must be open, in a separate terminal:

```bash
./infra/tunnel.sh
```

Read-only throughout — the `ecowitt_ro` role cannot write even by accident.

In [3]:
# Preflight: make this notebook work whatever kernel it lands on.
#
# VS Code writes the selected kernel back INTO this file, so pinning a
# kernelspec here loses to whatever was last used. This machine also has
# several Pythons (Anaconda, other project venvs). Rather than fight that,
# fall back to the project venv's site-packages when the current interpreter
# lacks the dependencies -- safe only when the Python versions match exactly,
# because compiled extensions like psycopg are ABI-specific.
import sys
from importlib.util import find_spec
from pathlib import Path

NEEDED = ("psycopg", "sqlalchemy", "pandas")


def _repo_venv_site_packages():
    """Walk up from the notebook looking for a sibling .venv."""
    for base in (Path.cwd(), *Path.cwd().parents):
        sp = base / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
        if sp.is_dir():
            return sp
    return None


missing = [m for m in NEEDED if find_spec(m) is None]
if missing:
    sp = _repo_venv_site_packages()
    if sp and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))
        missing = [m for m in NEEDED if find_spec(m) is None]
        if not missing:
            print(f"NOTE: running on {sys.executable}")
            print(f"      which lacks {', '.join(NEEDED)}, so this notebook added")
            print(f"      {sp}")
            print("      to sys.path as a fallback. It works because both interpreters are")
            print(f"      Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}.")
            print("      Cleaner: select the 'Python (ecowitt)' kernel (VS Code: kernel name,")
            print("      top-right of the notebook).")

if missing:
    raise SystemExit(
        f"Wrong kernel, and the fallback could not help.\n\n"
        f"  running on : {sys.executable}\n"
        f"  missing    : {', '.join(missing)}\n\n"
        "Select the 'Python (ecowitt)' kernel.\n"
        "  VS Code    : click the kernel name in the notebook's TOP-RIGHT corner\n"
        "               -> Select Another Kernel... -> Jupyter Kernel...\n"
        "  JupyterLab : Kernel > Change Kernel\n\n"
        "If it is not offered, register it once from the repo root:\n"
        "    .venv/bin/python -m ipykernel install --user --name ecowitt \\\n"
        "        --display-name 'Python (ecowitt)'"
    )

print(f"kernel OK: {sys.executable}")


NOTE: running on /Users/marcalexander/projects/evidence_project/.venv/bin/python
      which lacks psycopg, sqlalchemy, pandas, so this notebook added
      /Users/marcalexander/ecowitt_weather/.venv/lib/python3.12/site-packages
      to sys.path as a fallback. It works because both interpreters are
      Python 3.12.3.
      Cleaner: select the 'Python (ecowitt)' kernel (VS Code: kernel name,
      top-right of the notebook).
kernel OK: /Users/marcalexander/projects/evidence_project/.venv/bin/python


In [4]:
import os
import subprocess
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import psycopg
from sqlalchemy import create_engine, text

HOST, PORT = '127.0.0.1', 5433
DB, USER   = 'ecowitt', 'ecowitt_ro'
SECRET     = 'ecowitt-readonly-password'

# Day boundaries are a human idea, so they are computed in console-local time.
# Storage stays UTC; only the display is shifted.
DISPLAY_TZ = ZoneInfo('America/Chicago')
SAMPLE_ROWS = 5

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)


def password() -> str:
    """Env var first, else Secret Manager. Never hardcoded in this notebook."""
    if pw := os.environ.get('ECOWITT_RO_PASSWORD'):
        return pw
    out = subprocess.run(
        ['gcloud', 'secrets', 'versions', 'access', 'latest', f'--secret={SECRET}'],
        capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f'could not read {SECRET}: {out.stderr.strip()[:200]}')
    return out.stdout.strip()


from urllib.parse import quote_plus

engine = create_engine(
    f'postgresql+psycopg://{USER}:{quote_plus(password())}@{HOST}:{PORT}/{DB}',
    connect_args={'connect_timeout': 10})

try:
    conn = engine.connect()
except Exception as exc:
    raise SystemExit(
        f'Cannot reach {HOST}:{PORT}.\n'
        'The tunnel is almost certainly not running. In another terminal:\n'
        '    ./infra/tunnel.sh\n'
        f'Original error: {str(exc).splitlines()[0]}') from None

print(pd.read_sql(text('select current_user, current_database(), now()'), conn).to_string(index=False))

current_user current_database                              now
  ecowitt_ro          ecowitt 2026-08-03 04:21:39.269542+00:00


## 1. Discover objects from the catalog

Nothing here is hardcoded. Every table, view, materialised view and partitioned
table in `public` is picked up at runtime, so a new object added later is
automatically in scope for the sampling below.

In [5]:
OBJECTS_SQL = '''
SELECT c.relname                       AS object_name,
       CASE c.relkind WHEN 'r' THEN 'table'
                      WHEN 'p' THEN 'partitioned table'
                      WHEN 'v' THEN 'view'
                      WHEN 'm' THEN 'materialized view'
       END                             AS kind,
       pg_size_pretty(pg_total_relation_size(c.oid)) AS size,
       obj_description(c.oid)          AS comment
  FROM pg_class c
  JOIN pg_namespace n ON n.oid = c.relnamespace
 WHERE n.nspname = 'public'
   AND c.relkind IN ('r', 'p', 'v', 'm')
 ORDER BY c.relkind, c.relname
'''

objects = pd.read_sql(text(OBJECTS_SQL), conn)
print(f'{len(objects)} objects discovered\n')
objects

9 objects discovered



,object_name,kind,size,comment
0,backup_log,table,48 kB,None
1,change_log,table,32 kB,None
2,metric_catalog,table,48 kB,None
3,observation,table,1112 kB,None
4,quarantine,table,32 kB,None
5,raw_payload,table,144 kB,None
6,run_log,table,64 kB,None
7,observation_wide,view,0 bytes,None
8,observation_wide_metric,view,0 bytes,None


## 2. Row counts

Counted per object rather than read from `pg_class.reltuples`, which is only a
planner estimate and can be stale or plain wrong on a young table.

In [6]:
counts = []
for row in objects.itertuples():
    try:
        n = conn.execute(text(f'SELECT count(*) FROM public."{row.object_name}"')).scalar()
        counts.append({'object_name': row.object_name, 'kind': row.kind,
                       'rows': n, 'error': None})
    except Exception as exc:
        conn.rollback()
        counts.append({'object_name': row.object_name, 'kind': row.kind,
                       'rows': None, 'error': str(exc).splitlines()[0][:80]})

pd.DataFrame(counts)

,object_name,kind,rows,error
0,backup_log,table,2,None
1,change_log,table,0,None
2,metric_catalog,table,42,None
3,observation,table,3159,None
4,quarantine,table,0,None
5,raw_payload,table,7,None
6,run_log,table,8,None
7,observation_wide,view,81,None
8,observation_wide_metric,view,81,None


## 3. Sample rows from every object

A failure on one object is reported and skipped rather than aborting the batch —
an empty table and a broken view should be distinguishable at a glance.

In [7]:
for row in objects.itertuples():
    label = f'{row.object_name}  ({row.kind})'
    print('=' * 100)
    print(label)
    print('=' * 100)
    try:
        df = pd.read_sql(
            text(f'SELECT * FROM public."{row.object_name}" LIMIT {SAMPLE_ROWS}'), conn)
    except Exception as exc:
        conn.rollback()
        print(f'  ERROR: {str(exc).splitlines()[0][:120]}\n')
        continue
    if df.empty:
        print('  (no rows)\n')
    else:
        display(df)
    print()

backup_log  (table)


,backup_id,finished_at,object_uri,bytes,observation_rows
0,1,2026-08-02 22:56:49.553759+00:00,gs://ecowitt-504320-ecowitt-backups/daily/ecow...,26446,0
1,2,2026-08-02 23:55:50.588847+00:00,gs://ecowitt-504320-ecowitt-backups/daily/ecow...,46619,1833



change_log  (table)
  (no rows)


metric_catalog  (table)


,metric,kind,canonical_unit,resample_rule,resamplable,notes
0,outdoor.temperature,instantaneous,ºF,mean,True,NaN
1,outdoor.humidity,instantaneous,%,mean,True,NaN
2,outdoor.feels_like,derived,ºF,recompute,True,from temperature + humidity
3,outdoor.app_temp,derived,ºF,recompute,True,apparent temperature
4,outdoor.dew_point,derived,ºF,recompute,True,NaN



observation  (table)


...

,run_id,trigger,mode,window_start_utc,window_end_utc,cycle_type,observed_delta_s,started_at,ended_at,status,rows_fetched,rows_inserted,rows_updated,rows_unchanged,rows_rejected,error_detail
0,be28a752-8b96-4126-a3d0-6f2f243057c0,scheduled,incremental,2026-08-02 19:35:00+00:00,2026-08-02 23:35:00+00:00,5min,NaN,2026-08-02 23:36:54.956601+00:00,2026-08-02 23:36:57.319459+00:00,failed,0,0,0,0,0,GeneratedAlways: cannot insert a non-DEFAULT v...
1,d5007d76-eef1-48a1-a0c7-21a01317b79c,scheduled,incremental,2026-08-02 19:50:00+00:00,2026-08-02 23:50:00+00:00,5min,300.0,2026-08-02 23:54:26.498583+00:00,2026-08-02 23:54:28.973354+00:00,succeeded,1833,1833,0,0,0,NaN
2,a87175f2-2223-477e-9248-985dca963373,scheduled,incremental,2026-08-02 19:55:00+00:00,2026-08-02 23:55:00+00:00,5min,300.0,2026-08-02 23:55:09.840173+00:00,2026-08-02 23:55:12.500160+00:00,succeeded,1794,0,0,1794,0,NaN
3,87c89318-64e9-47ac-b349-7288699774e9,scheduled,incremental,2026-08-02 20:00:00+00:00,2026-08-03 00:00:00+00:00,5min,300.0,2026-08-03 00:02:28.630293+00:00,2026-08-03 00:02:31.000652+00:00,succeeded,1833,78,0,1755,0,NaN
4,5633f9ea-f4b7-4d02-8d98-ed003fede4ba,scheduled,incremental,2026-08-02 21:00:00+00:00,2026-08-03 01:00:00+00:00,5min,300.0,2026-08-03 01:01:34.317449+00:00,2026-08-03 01:01:36.973287+00:00,succeeded,1794,429,0,1365,0,NaN



observation_wide  (view)


,station_id,ts_utc,outdoor_temperature_f,outdoor_humidity_pct,outdoor_dew_point_f,pressure_absolute_inhg,wind_speed_mph,wind_gust_mph,wind_direction_deg,rain_rate_in_hr,solar_wm2,soil_ch1_pct,soil_ch2_pct,has_synthesized
0,24:4C:AB:74:9E:AA,2026-08-02 19:50:00+00:00,90.9,47.0,68.0,29.27,3.0,4.7,337.0,0.0,873.9,21.0,18.0,False
1,24:4C:AB:74:9E:AA,2026-08-02 19:55:00+00:00,91.9,47.0,69.0,29.26,4.5,7.4,340.0,0.0,897.0,21.0,18.0,False
2,24:4C:AB:74:9E:AA,2026-08-02 20:00:00+00:00,91.8,45.0,67.4,29.26,4.6,7.4,342.0,0.0,910.3,21.0,18.0,False
3,24:4C:AB:74:9E:AA,2026-08-02 20:05:00+00:00,90.1,47.0,67.4,29.26,4.0,6.5,334.0,0.0,840.3,21.0,18.0,False
4,24:4C:AB:74:9E:AA,2026-08-02 20:10:00+00:00,90.5,49.0,68.7,29.26,4.2,7.4,338.0,0.0,829.1,21.0,18.0,False



observation_wide_metric  (view)


,station_id,ts_utc,outdoor_temperature_c,outdoor_dew_point_c,pressure_absolute_hpa,wind_speed_ms,wind_gust_ms,rain_rate_mm_hr,outdoor_humidity_pct,wind_direction_deg,solar_wm2,has_synthesized
0,24:4C:AB:74:9E:AA,2026-08-02 19:50:00+00:00,32.722222,20.000000,991.196353,1.341120,2.101088,0.0,47.0,337.0,873.9,False
1,24:4C:AB:74:9E:AA,2026-08-02 19:55:00+00:00,33.277778,20.555556,990.857714,2.011680,3.308096,0.0,47.0,340.0,897.0,False
2,24:4C:AB:74:9E:AA,2026-08-02 20:00:00+00:00,33.222222,19.666667,990.857714,2.056384,3.308096,0.0,45.0,342.0,910.3,False
3,24:4C:AB:74:9E:AA,2026-08-02 20:05:00+00:00,32.277778,19.666667,990.857714,1.788160,2.905760,0.0,47.0,334.0,840.3,False
4,24:4C:AB:74:9E:AA,2026-08-02 20:10:00+00:00,32.500000,20.388889,990.857714,1.877568,3.308096,0.0,49.0,338.0,829.1,False


## 4. Hourly metric table — yesterday + today

Rows are hours, columns are metrics.

**Each metric is aggregated by its own rule from `metric_catalog`, not by a
blanket average.** That is not fussiness:

| kind | rule | why a plain average is wrong |
|---|---|---|
| `circular` | vector mean | averaging 350° and 10° linearly gives 180° — due south for a north wind. Measured 17 north-crossings in a single day of this station's data. |
| `extremum` | `max` | a gust is a per-interval maximum; averaging understates peak wind |
| `accumulator` | `last` | running totals that reset; averaging them is meaningless |
| `opaque` / `status` | `last` | not measurements; no arithmetic is valid |

The aggregation happens in SQL so the rules stay next to the data that defines them.

In [8]:
HOURLY_SQL = '''
WITH bounded AS (
    SELECT o.ts_utc, o.metric, o.value_num, c.resample_rule
      FROM observation o
      JOIN metric_catalog c USING (metric)
     WHERE o.ts_utc >= :start
       AND o.ts_utc <  :end
       AND o.value_num IS NOT NULL
),
hourly AS (
    SELECT date_trunc('hour', ts_utc AT TIME ZONE :tz) AS hour_local,
           metric,
           resample_rule,
           avg(value_num)                                  AS v_mean,
           max(value_num)                                  AS v_max,
           (array_agg(value_num ORDER BY ts_utc DESC))[1]  AS v_last,
           -- vector mean: atan2 of the mean sine and mean cosine
           atan2d(avg(sind(value_num)), avg(cosd(value_num))) AS v_vector
      FROM bounded
     GROUP BY 1, 2, 3
)
SELECT hour_local,
       metric,
       CASE resample_rule
           WHEN 'max'         THEN v_max
           WHEN 'last'        THEN v_last
           WHEN 'carry'       THEN v_last
           WHEN 'vector_mean' THEN CASE WHEN v_vector < 0 THEN v_vector + 360
                                        ELSE v_vector END
           ELSE v_mean
       END AS value
  FROM hourly
 ORDER BY hour_local, metric
'''

now_local   = datetime.now(DISPLAY_TZ)
start_local = (now_local - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
end_local   = (now_local + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)

print(f'window: {start_local:%Y-%m-%d %H:%M %Z} -> {end_local:%Y-%m-%d %H:%M %Z}')

long = pd.read_sql(text(HOURLY_SQL), conn, params={
    'start': start_local, 'end': end_local, 'tz': str(DISPLAY_TZ)})
print(f'{len(long):,} metric-hours across {long.metric.nunique()} metrics')

window: 2026-08-01 00:00 CDT -> 2026-08-03 00:00 CDT
351 metric-hours across 39 metrics


In [9]:
hourly = (long.pivot(index='hour_local', columns='metric', values='value')
              .sort_index())
hourly.index.name = f'hour ({DISPLAY_TZ})'

# Empty columns mean the metric reported nothing in the window -- worth seeing,
# not worth hiding.
print(f'{hourly.shape[0]} hours x {hourly.shape[1]} metrics')
hourly.round(2)

9 hours x 39 metrics


metric,battery.haptic_array_battery,battery.haptic_array_capacitor,battery.soilmoisture_sensor_ch1,battery.soilmoisture_sensor_ch2,indoor.app_tempin,indoor.dew_point,indoor.feels_like,indoor.humidity,indoor.temperature,outdoor.app_temp,outdoor.dew_point,outdoor.feels_like,outdoor.humidity,outdoor.temperature,outdoor.vpd,pressure.absolute,pressure.relative,rainfall_piezo.1_hour,rainfall_piezo.daily,rainfall_piezo.event,rainfall_piezo.monthly,rainfall_piezo.rain_rate,rainfall_piezo.weekly,rainfall_piezo.yearly,soil_ch1.ad,soil_ch1.soilmoisture,soil_ch2.ad,soil_ch2.soilmoisture,solar_and_uvi.solar,solar_and_uvi.uvi,temp_and_humidity_ch1.humidity,temp_and_humidity_ch1.temperature,temp_and_humidity_ch2.humidity,temp_and_humidity_ch2.temperature,temp_and_humidity_ch3.humidity,temp_and_humidity_ch3.temperature,wind.wind_direction,wind.wind_gust,wind.wind_speed
hour (America/Chicago),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-02 14:00:00,3.28,5.3,1.6,1.6,77.10,53.40,76.05,45.50,76.05,96.15,68.50,95.95,47.00,91.40,0.78,29.26,29.26,0.0,0.0,0.0,0.0,0.0,0.0,0.05,147.0,21.0,129.0,18.0,885.45,8.00,50.50,76.30,48.00,75.80,45.50,75.75,338.50,7.4,3.75
2026-08-02 15:00:00,3.28,5.3,1.6,1.6,76.97,53.09,76.01,44.92,76.01,95.90,67.99,95.79,46.17,91.61,0.81,29.26,29.26,0.0,0.0,0.0,0.0,0.0,0.0,0.05,147.0,21.0,129.0,18.0,790.28,6.83,50.67,76.47,48.00,75.94,44.75,76.08,335.41,7.4,4.14
2026-08-02 16:00:00,3.26,5.3,1.6,1.6,76.63,52.96,75.72,45.17,75.72,95.84,67.24,95.60,44.58,91.88,0.84,29.24,29.24,0.0,0.0,0.0,0.0,0.0,0.0,0.05,147.0,21.0,129.0,18.0,659.78,5.58,50.33,76.66,48.00,76.16,44.58,76.01,333.67,8.7,4.12
2026-08-02 17:00:00,3.28,5.3,1.6,1.6,75.29,52.22,74.58,45.67,74.58,93.63,66.19,92.38,46.25,89.65,0.76,29.23,29.23,0.0,0.0,0.0,0.0,0.0,0.0,0.05,147.0,21.0,129.0,18.0,260.25,1.92,49.67,76.70,47.33,76.25,45.00,74.91,333.08,8.9,3.26
2026-08-02 18:00:00,3.28,5.3,1.6,1.6,74.41,51.71,73.87,46.00,73.87,91.95,65.39,89.87,47.73,87.81,0.70,29.22,29.22,0.0,0.0,0.0,0.0,0.0,0.0,0.05,146.0,21.0,129.0,18.0,73.06,0.09,47.82,76.45,45.73,75.92,45.18,74.26,336.46,6.0,2.33
2026-08-02 19:00:00,3.28,5.3,1.6,1.6,74.50,51.37,74.05,44.92,74.05,90.98,64.72,87.98,48.67,86.31,0.65,29.21,29.21,0.0,0.0,0.0,0.0,0.0,0.0,0.05,146.0,21.0,129.0,18.0,35.18,0.00,46.75,76.09,44.50,75.62,44.33,74.37,338.10,3.6,0.86
2026-08-02 20:00:00,3.28,5.2,1.6,1.6,74.20,51.26,73.76,45.36,73.76,88.90,65.90,85.35,56.09,83.25,0.51,29.21,29.21,0.0,0.0,0.0,0.0,0.0,0.0,0.05,146.0,21.0,129.0,18.0,4.09,0.00,46.64,75.68,44.36,75.45,44.27,74.36,162.72,1.3,0.00
2026-08-02 21:00:00,3.28,5.2,1.6,1.6,75.00,51.90,74.41,45.29,74.41,85.83,66.33,81.03,63.14,80.00,0.38,29.22,29.22,0.0,0.0,0.0,0.0,0.0,0.0,0.05,146.0,21.0,129.0,18.0,0.00,0.00,47.14,75.50,44.71,75.41,45.00,74.69,278.88,1.3,0.00
2026-08-02 22:00:00,3.28,5.1,1.6,1.6,75.10,52.00,74.45,45.50,74.45,84.05,67.55,77.65,71.00,77.65,0.28,29.23,29.23,0.0,0.0,0.0,0.0,0.0,0.0,0.05,146.0,21.0,129.0,18.0,0.00,0.00,48.00,75.40,45.00,75.45,44.00,75.50,315.50,1.1,0.15


### Weather-only view

The full table includes battery voltages and status codes. This drops the
diagnostic and status metrics to leave the actual weather.

In [10]:
WEATHER_ONLY = '''
SELECT metric FROM metric_catalog
 WHERE kind NOT IN ('diagnostic', 'status', 'opaque')
 ORDER BY metric
'''
weather = pd.read_sql(text(WEATHER_ONLY), conn).metric.tolist()
cols = [c for c in hourly.columns if c in weather]
hourly[cols].round(2)

metric,indoor.app_tempin,indoor.dew_point,indoor.feels_like,indoor.humidity,indoor.temperature,outdoor.app_temp,outdoor.dew_point,outdoor.feels_like,outdoor.humidity,outdoor.temperature,outdoor.vpd,pressure.absolute,pressure.relative,rainfall_piezo.1_hour,rainfall_piezo.daily,rainfall_piezo.event,rainfall_piezo.monthly,rainfall_piezo.rain_rate,rainfall_piezo.weekly,rainfall_piezo.yearly,soil_ch1.soilmoisture,soil_ch2.soilmoisture,solar_and_uvi.solar,solar_and_uvi.uvi,temp_and_humidity_ch1.humidity,temp_and_humidity_ch1.temperature,temp_and_humidity_ch2.humidity,temp_and_humidity_ch2.temperature,temp_and_humidity_ch3.humidity,temp_and_humidity_ch3.temperature,wind.wind_direction,wind.wind_gust,wind.wind_speed
hour (America/Chicago),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-02 14:00:00,77.10,53.40,76.05,45.50,76.05,96.15,68.50,95.95,47.00,91.40,0.78,29.26,29.26,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,885.45,8.00,50.50,76.30,48.00,75.80,45.50,75.75,338.50,7.4,3.75
2026-08-02 15:00:00,76.97,53.09,76.01,44.92,76.01,95.90,67.99,95.79,46.17,91.61,0.81,29.26,29.26,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,790.28,6.83,50.67,76.47,48.00,75.94,44.75,76.08,335.41,7.4,4.14
2026-08-02 16:00:00,76.63,52.96,75.72,45.17,75.72,95.84,67.24,95.60,44.58,91.88,0.84,29.24,29.24,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,659.78,5.58,50.33,76.66,48.00,76.16,44.58,76.01,333.67,8.7,4.12
2026-08-02 17:00:00,75.29,52.22,74.58,45.67,74.58,93.63,66.19,92.38,46.25,89.65,0.76,29.23,29.23,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,260.25,1.92,49.67,76.70,47.33,76.25,45.00,74.91,333.08,8.9,3.26
2026-08-02 18:00:00,74.41,51.71,73.87,46.00,73.87,91.95,65.39,89.87,47.73,87.81,0.70,29.22,29.22,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,73.06,0.09,47.82,76.45,45.73,75.92,45.18,74.26,336.46,6.0,2.33
2026-08-02 19:00:00,74.50,51.37,74.05,44.92,74.05,90.98,64.72,87.98,48.67,86.31,0.65,29.21,29.21,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,35.18,0.00,46.75,76.09,44.50,75.62,44.33,74.37,338.10,3.6,0.86
2026-08-02 20:00:00,74.20,51.26,73.76,45.36,73.76,88.90,65.90,85.35,56.09,83.25,0.51,29.21,29.21,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,4.09,0.00,46.64,75.68,44.36,75.45,44.27,74.36,162.72,1.3,0.00
2026-08-02 21:00:00,75.00,51.90,74.41,45.29,74.41,85.83,66.33,81.03,63.14,80.00,0.38,29.22,29.22,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,0.00,0.00,47.14,75.50,44.71,75.41,45.00,74.69,278.88,1.3,0.00
2026-08-02 22:00:00,75.10,52.00,74.45,45.50,74.45,84.05,67.55,77.65,71.00,77.65,0.28,29.23,29.23,0.0,0.0,0.0,0.0,0.0,0.0,0.05,21.0,18.0,0.00,0.00,48.00,75.40,45.00,75.45,44.00,75.50,315.50,1.1,0.15


### Coverage

How many 5-minute observations landed in each hour. A full hour is 12.
Anything less is a dropout — routine on this station, so this is a health
check rather than an alarm.

In [11]:
COVERAGE_SQL = '''
SELECT date_trunc('hour', ts_utc AT TIME ZONE :tz) AS hour_local,
       count(DISTINCT ts_utc)                         AS slots,
       round(100.0 * count(DISTINCT ts_utc) / 12, 1)  AS pct_of_expected
  FROM observation
 WHERE ts_utc >= :start AND ts_utc < :end
 GROUP BY 1 ORDER BY 1
'''
cov = pd.read_sql(text(COVERAGE_SQL), conn, params={
    'start': start_local, 'end': end_local, 'tz': str(DISPLAY_TZ)})
# The first and last hours are partial by construction -- the window edge cuts
# them, and before ingestion started there is simply no data. Including them
# reports a dropout that never happened, so they are excluded from the summary
# but kept in the table.
full = cov.iloc[1:-1] if len(cov) > 2 else cov
if len(full):
    print(f'mean coverage (excluding partial edge hours): {full.pct_of_expected.mean():.1f}%')
    print(f'hours below 100%: {(full.slots < 12).sum()} of {len(full)}')
print('first/last rows are edge hours -- partial by construction, not dropouts')
cov

mean coverage (excluding partial edge hours): 91.7%
hours below 100%: 3 of 7
first/last rows are edge hours -- partial by construction, not dropouts


,hour_local,slots,pct_of_expected
0,2026-08-02 14:00:00,2,16.7
1,2026-08-02 15:00:00,12,100.0
2,2026-08-02 16:00:00,12,100.0
3,2026-08-02 17:00:00,12,100.0
4,2026-08-02 18:00:00,11,91.7
5,2026-08-02 19:00:00,12,100.0
6,2026-08-02 20:00:00,11,91.7
7,2026-08-02 21:00:00,7,58.3
8,2026-08-02 22:00:00,2,16.7


In [12]:
conn.close()
engine.dispose()
print('connection closed')

connection closed
